# Random Forest và XGBoost (cửa sổ 28 ngày)
Dự đoán `label_withdrawn` = 1 cho sinh viên còn học ở đầu ngày 28, từ thông tin ngày 0–27. Fit trên train, đánh giá baseline trên dev ở ngưỡng 0,5. Không dùng test ở giai đoạn này.

Hai mô hình dùng cùng 33 cột gốc và nhánh `tree` của preprocessor chung (49 feature sau biến đổi). Mã huấn luyện dùng lại nằm trong `src/models/baseline_rf_xgb_real.py`.

**Dữ liệu:** 19.264 lượt đăng ký thuộc tập train; 4.131 lượt thuộc dev. `feature_28d.csv`, `split.csv` và `preprocessor.joblib` nằm trong `dataset/`. Đường dẫn dữ liệu giữ nguyên như baseline hiện tại.

In [1]:
import sys
import warnings
from pathlib import Path
from sklearn.exceptions import InconsistentVersionWarning

# Artifact được lưu bằng scikit-learn 1.9.0; chỉ ẩn cảnh báo phiên bản khi đọc.
warnings.filterwarnings('ignore', category=InconsistentVersionWarning)
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.models.baseline_rf_xgb_real import run
DATASET = ROOT / 'dataset'
RESULTS = ROOT / 'results'
assert DATASET.is_dir(), f'Không tìm thấy thư mục {DATASET}'

## Baseline trên train và dev
RF và XGBoost dùng cấu hình cố định 100 cây. Hai mô hình nhận cùng dữ liệu sau biến đổi; đây là mốc ban đầu trước khi tinh chỉnh tham số và ngưỡng.

In [2]:
metrics, importance, raw_features = run(DATASET, RESULTS)
print(f'{len(raw_features)} cột gốc → {metrics.n_features.iloc[0]} feature cho mỗi mô hình')
metrics

33 cột gốc → 49 feature cho mỗi mô hình


,model,split,n_train,n_dev,n_features,threshold,precision,recall,f1,roc_auc,pr_auc,tn,fp,fn,tp
0,random_forest,dev,19264,4131,49,0.5,0.422139,0.299601,0.350467,0.694753,0.353224,3072,308,526,225
1,xgboost,dev,19264,4131,49,0.5,0.310954,0.585885,0.406279,0.704632,0.378043,2405,975,311,440


**Nhận xét trên dev:** RF có precision khoảng 0,42 nhưng recall khoảng 0,30; XGBoost có recall khoảng 0,59 và precision khoảng 0,31. F1 tương ứng khoảng 0,35 và 0,41. Đây là kết quả tại ngưỡng 0,5, chưa phải kết luận sau tinh chỉnh. Các metric được tính cho lớp dương **Withdrawn**.

## Feature importance
Bảng dưới đây liệt kê 10 feature có importance lớn nhất của mỗi mô hình. Tên cột là tên sau mã hóa; importance mô tả mức mô hình sử dụng feature, không chứng minh quan hệ nhân quả.

In [3]:
importance.groupby('model', group_keys=False).head(10)

,model,feature,importance
36,random_forest,num_raw__date_registration,0.067339
43,random_forest,num_raw__avg_score,0.055542
25,random_forest,num_raw__clicks_per_active_day,0.054068
37,random_forest,num_raw__click_trend,0.054034
23,random_forest,num_raw__total_clicks,0.053034
28,random_forest,num_raw__clicks_w3,0.047480
26,random_forest,num_raw__clicks_w1,0.046912
27,random_forest,num_raw__clicks_w2,0.046267
29,random_forest,num_raw__clicks_w4,0.044607
30,random_forest,num_raw__n_distinct_sites,0.042955


**Đầu ra:** `results/rf_xgb_baseline_dev.csv` và `results/rf_xgb_feature_importance.csv`. Bản baseline 68 feature trước khi dùng preprocessor chung được lưu riêng với hậu tố `legacy_68`.

**Tiếp theo:** Stratified Group K-fold trong train, chọn cấu hình/ngưỡng trên dev, thực nghiệm PCA và ablation; chỉ đánh giá test sau khi khóa lựa chọn. LR dùng nhánh `linear` của preprocessor, còn hai mô hình cây dùng nhánh `tree`, nên cách biến đổi số khác nhau theo loại mô hình.